In [ ]:
# ============================================================
# Mount Google Drive
# ============================================================

from google.colab import drive
import os

drive.mount("/content/drive")

print("\nChecking Google Drive...")

if os.path.exists("/content/drive/MyDrive"):
    print("✅ Google Drive mounted successfully.")
    print("MyDrive path:")
    print("/content/drive/MyDrive")
else:
    raise RuntimeError("❌ Google Drive was not mounted correctly.")

In [ ]:
import os

print("Experiment directory:")
print(EXPERIMENT_DIR)

os.makedirs(EXPERIMENT_DIR, exist_ok=True)

print("\nContents:")

if os.path.exists(EXPERIMENT_DIR):
    for item in os.listdir(EXPERIMENT_DIR):
        print("-", item)
else:
    print("Directory does not exist.")

In [1]:
# ============================================================
# Cell 1 - Imports & Reproducibility
# ============================================================

import os
import random
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import transforms
import timm

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 60)
print(f"PyTorch Version : {torch.__version__}")
print(f"Device          : {device}")
print("=" * 60)

PyTorch Version : 2.11.0+cu128
Device          : cuda


In [2]:
# ============================================================
# Cell 2 - Download & Extract Dataset
# ============================================================

import os
import zipfile
import subprocess
from pathlib import Path

DATA_ROOT = Path("/content/HER2_Dataset")
DATA_ROOT.mkdir(exist_ok=True)

ZIP_PATH = DATA_ROOT / "her2-ihc-40x-wsi.zip"

URL = "https://zenodo.org/records/15179608/files/her2-ihc-40x-wsi.zip?download=1"

# ------------------------------------------------------------
# Download dataset (only if not already downloaded)
# ------------------------------------------------------------

if not ZIP_PATH.exists():
    print("Downloading HER2-IHC-40x dataset...")
    subprocess.run([
        "wget",
        "-O",
        str(ZIP_PATH),
        URL
    ], check=True)
else:
    print("Dataset archive already exists.")

# ------------------------------------------------------------
# Extract main archive
# ------------------------------------------------------------

WSI_DIR = DATA_ROOT / "WSI-based-dataset"

if not WSI_DIR.exists():
    print("Extracting main archive...")

    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATA_ROOT)

    print("Main archive extracted.")

else:
    print("Main archive already extracted.")

# ------------------------------------------------------------
# Extract nested train/test archives
# ------------------------------------------------------------

nested_archives = [
    WSI_DIR / "train_data_wsi.zip",
    WSI_DIR / "test_data_wsi.zip",
]

for archive in nested_archives:

    extract_folder = archive.parent / archive.stem.replace("_data_wsi", "")

    if extract_folder.exists():
        print(f"{extract_folder.name} already extracted.")
        continue

    print(f"Extracting {archive.name}...")

    with zipfile.ZipFile(archive, "r") as z:
        z.extractall(extract_folder)

# ------------------------------------------------------------
# Remove zip archives to save disk space
# ------------------------------------------------------------

for archive in [ZIP_PATH] + nested_archives:
    if archive.exists():
        archive.unlink()

print("ZIP files removed.")

# ------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------

TRAIN_DIR = WSI_DIR / "train"
TEST_DIR = WSI_DIR / "test"

print("\nDataset location:")
print(TRAIN_DIR)
print(TEST_DIR)

assert TRAIN_DIR.exists(), "Train directory not found."
assert TEST_DIR.exists(), "Test directory not found."

print("\nDataset successfully prepared!")

# ------------------------------------------------------------
# Show directory structure
# ------------------------------------------------------------

print("\nFolder structure:")

for folder in [TRAIN_DIR, TEST_DIR]:
    print(f"\n{folder.name}/")

    for cls in sorted(os.listdir(folder)):
        cls_path = folder / cls
        if cls_path.is_dir():
            n = len(os.listdir(cls_path))
            print(f"   {cls:<10} {n:5d} images")

Extracting main archive...
Main archive extracted.
Extracting train_data_wsi.zip...
Extracting test_data_wsi.zip...
ZIP files removed.

Dataset location:
/content/HER2_Dataset/WSI-based-dataset/train
/content/HER2_Dataset/WSI-based-dataset/test

Dataset successfully prepared!

Folder structure:

train/
   class_0     3131 images
   class_1+    1837 images
   class_2+     523 images
   class_3+    2602 images

test/
   class_0      658 images
   class_1+     316 images
   class_2+     111 images
   class_3+     762 images


In [3]:
# ============================================================
# Cell 3 - HER2 Dataset
# ============================================================

class HER2Dataset(Dataset):
    """
    HER2-IHC-40x Dataset

    Expected directory structure:

    train/
        class_0/
        class_1+/
        class_2+/
        class_3+

    test/
        class_0/
        class_1+/
        class_2+/
        class_3+
    """

    EXPECTED_CLASSES = [
        "class_0",
        "class_1+",
        "class_2+",
        "class_3+"
    ]

    IMAGE_EXTENSIONS = {
        ".png",
        ".jpg",
        ".jpeg"
    }

    def __init__(self, root_dir, transform=None):

        self.root_dir = Path(root_dir)
        self.transform = transform

        if not self.root_dir.exists():
            raise FileNotFoundError(
                f"Directory not found:\n{self.root_dir}"
            )

        # ----------------------------------------------------
        # Verify class folders
        # ----------------------------------------------------

        self.classes = sorted([
            d.name
            for d in self.root_dir.iterdir()
            if d.is_dir() and not d.name.startswith(".")
        ])

        if self.classes != self.EXPECTED_CLASSES:
            raise ValueError(
                f"Expected folders:\n"
                f"{self.EXPECTED_CLASSES}\n\n"
                f"Found:\n"
                f"{self.classes}"
            )

        self.class_to_idx = {
            cls: idx
            for idx, cls in enumerate(self.classes)
        }

        self.idx_to_class = {
            idx: cls
            for cls, idx in self.class_to_idx.items()
        }

        # ----------------------------------------------------
        # Collect image paths
        # ----------------------------------------------------

        self.samples = []

        for cls in self.classes:

            class_dir = self.root_dir / cls

            images = sorted([
                p
                for p in class_dir.iterdir()
                if p.suffix.lower() in self.IMAGE_EXTENSIONS
            ])

            if len(images) == 0:
                raise RuntimeError(
                    f"No images found in {class_dir}"
                )

            label = self.class_to_idx[cls]

            for img_path in images:
                self.samples.append(
                    (img_path, label)
                )

        print(f"Loaded {len(self.samples)} images from {self.root_dir.name}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):

        img_path, label = self.samples[index]

        image = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label

    def get_num_classes(self):
        return len(self.classes)

    def get_class_names(self):
        return self.classes

    def get_class_distribution(self):

        distribution = {
            cls: 0
            for cls in self.classes
        }

        for _, label in self.samples:
            cls = self.idx_to_class[label]
            distribution[cls] += 1

        return distribution

In [4]:
# ============================================================
# Cell 4 - Image Transforms
# ============================================================

# ImageNet normalization (DenseNet121 is ImageNet pretrained)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# ------------------------------------------------------------
# Training Transform
# Paper:
# Resize
# Horizontal Flip
# Vertical Flip
# Rotation
# Normalization
# ------------------------------------------------------------

train_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomVerticalFlip(p=0.5),

    transforms.RandomRotation(
        degrees=10,
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=0
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# ------------------------------------------------------------
# Validation / Test Transform
# No augmentation
# ------------------------------------------------------------

test_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

print("Transforms created successfully.")

Transforms created successfully.


In [5]:
# ==========================================================
# Cell 6 — Model (Stage 1) - Vision Transformer
# ==========================================================

import torch
import torch.nn as nn
import torch.optim as optim
import timm

# ----------------------------------------------------------
# Load ImageNet pretrained ViT-B/16
# ----------------------------------------------------------

model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=True,
    num_classes=4
)

model = model.to(device)

# ----------------------------------------------------------
# Stage 1
# Freeze ViT backbone
# Train ONLY classification head
# ----------------------------------------------------------

for name, param in model.named_parameters():

    if "head" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

print("Stage 1 configuration")
print("---------------------")
print("Backbone : Frozen")
print("Head : Trainable")

# ----------------------------------------------------------
# Loss
# ----------------------------------------------------------

criterion = nn.CrossEntropyLoss()

# ----------------------------------------------------------
# Optimizer
# ----------------------------------------------------------

optimizer = optim.Adam(
    model.head.parameters(),
    lr=1e-4
)

# ----------------------------------------------------------
# Scheduler
# ----------------------------------------------------------

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30
)

print("\nModel initialized successfully.")

print(model)

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Stage 1 configuration
---------------------
Backbone : Frozen
Head : Trainable

Model initialized successfully.
VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out

In [6]:
# ============================================================
# Cell 7 - Training & Evaluation Functions
# ============================================================

from sklearn.metrics import accuracy_score

# ------------------------------------------------------------
# Train one epoch
# ------------------------------------------------------------

def train_one_epoch(model, dataloader, criterion, optimizer):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc


# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

def evaluate(model, dataloader, criterion):

    model.eval()

    running_loss = 0.0

    preds_all = []
    labels_all = []

    with torch.no_grad():

        for images, labels in dataloader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)

    epoch_acc = accuracy_score(labels_all, preds_all) * 100

    return (
        epoch_loss,
        epoch_acc,
        preds_all,
        labels_all
    )

In [7]:
# ============================================================
# Create datasets
# ============================================================

train_dataset = HER2Dataset(
    root_dir=TRAIN_DIR,
    transform=train_transform
)

val_dataset = HER2Dataset(
    root_dir=TEST_DIR,
    transform=test_transform
)

print("Train images :", len(train_dataset))
print("Validation images :", len(val_dataset))

Loaded 8093 images from train
Loaded 1847 images from test
Train images : 8093
Validation images : 1847


In [8]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(len(train_loader), len(val_loader))

253 58


In [9]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Train batches: 253
Validation batches: 58


In [10]:
# ============================================================
# Experiment Configuration
# ============================================================

# Backbone information
BACKBONE_NAME = "ViT_B16"
MODEL_ID = "vit_base_patch16_224"

NUM_CLASSES = 4
IMAGE_SIZE = 224

# Stage folders
CHECKPOINT_ROOT = "/content/drive/MyDrive/HER2_Checkpoints"
EXPERIMENT_DIR = os.path.join(CHECKPOINT_ROOT, BACKBONE_NAME)

print("=" * 60)
print("Experiment Configuration")
print("=" * 60)
print(f"Backbone : {BACKBONE_NAME}")
print(f"Model ID : {MODEL_ID}")
print(f"Save Dir : {EXPERIMENT_DIR}")
print("=" * 60)

Experiment Configuration
Backbone : ViT_B16
Model ID : vit_base_patch16_224
Save Dir : /content/drive/MyDrive/HER2_Checkpoints/ViT_B16


In [11]:
# ============================================================
# Parameter Group Utilities
# ============================================================

def get_parameter_groups(model, backbone_name):
    """
    Split model parameters into:
        1. Backbone parameters
        2. Classification head parameters

    Returns:
        backbone_params, head_params
    """

    backbone_name = backbone_name.lower()

    # --------------------------------------------------------
    # Vision Transformer (timm)
    # --------------------------------------------------------
    if "vit" in backbone_name:

        head_keywords = ["head"]

    # --------------------------------------------------------
    # DenseNet
    # --------------------------------------------------------
    elif "densenet" in backbone_name:

        head_keywords = ["classifier"]

    # --------------------------------------------------------
    # ConvNeXt
    # --------------------------------------------------------
    elif "convnext" in backbone_name:

        head_keywords = ["head", "classifier"]

    # --------------------------------------------------------
    # Swin Transformer
    # --------------------------------------------------------
    elif "swin" in backbone_name:

        head_keywords = ["head"]

    # --------------------------------------------------------
    # EfficientNet
    # --------------------------------------------------------
    elif "efficientnet" in backbone_name:

        head_keywords = ["classifier"]

    # --------------------------------------------------------
    # ResNet
    # --------------------------------------------------------
    elif "resnet" in backbone_name:

        head_keywords = ["fc"]

    else:
        raise ValueError(
            f"Unsupported backbone: {backbone_name}"
        )

    backbone_params = []
    head_params = []

    for name, param in model.named_parameters():

        if any(keyword in name for keyword in head_keywords):
            head_params.append(param)
        else:
            backbone_params.append(param)

    return backbone_params, head_params


In [12]:
# ============================================================
# CELL 7 — STAGE 1 : Train Classification Head Only
# ============================================================

# ------------------------------------------------------------
# Local checkpoint path (fast saving)
# ------------------------------------------------------------

BEST_MODEL_PATH = f"/content/best_stage1_{BACKBONE_NAME}.pth"

# ------------------------------------------------------------
# Get parameter groups
# ------------------------------------------------------------

backbone_params, head_params = get_parameter_groups(
    model,
    BACKBONE_NAME
)

# ------------------------------------------------------------
# Freeze backbone
# ------------------------------------------------------------

for param in backbone_params:
    param.requires_grad = False

for param in head_params:
    param.requires_grad = True

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = optim.Adam(
    head_params,
    lr=1e-4
)

# ------------------------------------------------------------
# Scheduler
# ------------------------------------------------------------

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30
)

criterion = nn.CrossEntropyLoss()

stage1_epochs = 30

best_acc = 0.0
best_epoch = 0

print("=" * 60)
print(f"Stage 1 - {BACKBONE_NAME}")
print("Training classification head only")
print("=" * 60)

for epoch in range(stage1_epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    val_loss, val_acc, preds, labels = evaluate(
        model,
        val_loader,
        criterion
    )

    scheduler.step()

    print(
        f"Epoch [{epoch+1:02d}/{stage1_epochs}] | "
        f"Train Loss {train_loss:.4f} | "
        f"Train Acc {train_acc:.2f}% | "
        f"Val Loss {val_loss:.4f} | "
        f"Val Acc {val_acc:.2f}%"
    )

    # --------------------------------------------------------
    # Save best checkpoint (LOCAL)
    # --------------------------------------------------------

    if val_acc > best_acc:

        best_acc = val_acc
        best_epoch = epoch + 1

        torch.save(
            {
                "stage": 1,
                "epoch": best_epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_accuracy": best_acc,
                "model_name": BACKBONE_NAME,
                "model_id": MODEL_ID,
                "library": "timm",
                "pretrained": True,
                "pretrained_dataset": "ImageNet-1K",
                "num_classes": 4,
            },
            BEST_MODEL_PATH,
        )

        print(
            f"✅ New best model saved "
            f"(Epoch {best_epoch} | Val Acc: {best_acc:.2f}%)"
        )

print("\n" + "=" * 60)
print("Stage 1 Finished")
print("=" * 60)
print(f"Best Accuracy : {best_acc:.2f}%")
print(f"Best Epoch    : {best_epoch}")
print(f"Local Model   : {BEST_MODEL_PATH}")

Stage 1 - ViT_B16
Training classification head only
Epoch [01/30] | Train Loss 0.6696 | Train Acc 76.12% | Val Loss 0.5763 | Val Acc 79.32%
✅ New best model saved (Epoch 1 | Val Acc: 79.32%)
Epoch [02/30] | Train Loss 0.3192 | Train Acc 90.23% | Val Loss 0.4468 | Val Acc 84.08%
✅ New best model saved (Epoch 2 | Val Acc: 84.08%)
Epoch [03/30] | Train Loss 0.2504 | Train Acc 92.10% | Val Loss 0.3997 | Val Acc 85.44%
✅ New best model saved (Epoch 3 | Val Acc: 85.44%)
Epoch [04/30] | Train Loss 0.2142 | Train Acc 93.51% | Val Loss 0.3829 | Val Acc 85.71%
✅ New best model saved (Epoch 4 | Val Acc: 85.71%)
Epoch [05/30] | Train Loss 0.1965 | Train Acc 93.65% | Val Loss 0.3609 | Val Acc 86.52%
✅ New best model saved (Epoch 5 | Val Acc: 86.52%)
Epoch [06/30] | Train Loss 0.1807 | Train Acc 93.91% | Val Loss 0.3440 | Val Acc 86.84%
✅ New best model saved (Epoch 6 | Val Acc: 86.84%)
Epoch [07/30] | Train Loss 0.1669 | Train Acc 94.56% | Val Loss 0.3523 | Val Acc 86.63%
Epoch [08/30] | Train Loss

In [13]:
import os
import shutil
from datetime import datetime

# ============================================================
# Save Stage 1 Results to Google Drive
# ============================================================

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

LOCAL_MODEL = f"/content/best_stage1_{BACKBONE_NAME}.pth"

SAVE_DIR = os.path.join(
    EXPERIMENT_DIR,
    "Stage1"
)
os.makedirs(SAVE_DIR, exist_ok=True)

checkpoint_filename = f"best_stage1_{BACKBONE_NAME}.pth"
summary_filename = f"stage1_summary_{BACKBONE_NAME}.txt"

DEST_MODEL = os.path.join(
    SAVE_DIR,
    checkpoint_filename
)

report_path = os.path.join(
    SAVE_DIR,
    summary_filename
)

# ------------------------------------------------------------
# Copy checkpoint
# ------------------------------------------------------------

if not os.path.exists(LOCAL_MODEL):
    raise FileNotFoundError(
        f"Checkpoint not found:\n{LOCAL_MODEL}"
    )

shutil.copy2(LOCAL_MODEL, DEST_MODEL)

print("✅ Best Stage 1 model copied successfully.")
print(f"Destination : {DEST_MODEL}")

size_mb = os.path.getsize(DEST_MODEL) / 1024 / 1024
print(f"Model size  : {size_mb:.2f} MB")

# ------------------------------------------------------------
# Generate experiment report
# ------------------------------------------------------------

summary = f"""
=========================================================
HER2-IHC-40x
Stage 1 Training Summary
=========================================================

Date
---------------------------------------------------------
{datetime.now()}

Experiment
---------------------------------------------------------
Backbone              : {BACKBONE_NAME}
Model ID              : {MODEL_ID}
Library               : timm
Pretrained            : True
Pretrained Dataset    : ImageNet-1K
Number of Classes     : 4

Training Strategy
---------------------------------------------------------
Stage                 : 1
Description           : Frozen backbone, train classification head only

Dataset
---------------------------------------------------------
Training Images       : {len(train_dataset)}
Validation Images     : {len(val_dataset)}
Classes               : {', '.join(train_dataset.get_class_names())}

Input
---------------------------------------------------------
Image Size            : 224 x 224

Optimization
---------------------------------------------------------
Optimizer             : Adam
Learning Rate         : 1e-4
Scheduler             : CosineAnnealingLR
Loss                  : CrossEntropyLoss
Batch Size            : {BATCH_SIZE}
Epochs                : {stage1_epochs}

Results
---------------------------------------------------------
Best Validation Acc   : {best_acc:.4f} %
Best Epoch            : {best_epoch}

Checkpoint
---------------------------------------------------------
Filename              : {checkpoint_filename}
Location              : {DEST_MODEL}

=========================================================
"""

with open(report_path, "w") as f:
    f.write(summary)

print(f"✅ Summary saved:\n{report_path}")

✅ Best Stage 1 model copied successfully.
Destination : /content/drive/MyDrive/HER2_Checkpoints/ViT_B16/Stage1/best_stage1_ViT_B16.pth
Model size  : 327.39 MB
✅ Summary saved:
/content/drive/MyDrive/HER2_Checkpoints/ViT_B16/Stage1/stage1_summary_ViT_B16.txt


# stage 2

In [14]:
# ============================================================
# Validation
# ============================================================

import torch

def validate_one_epoch(
    model,
    dataloader,
    criterion,
):
    """
    Validate the model for one epoch.

    Returns
    -------
    epoch_loss : float
    epoch_acc  : float
    predictions: list[int]
    labels     : list[int]
    """

    model.eval()

    running_loss = 0.0

    predictions = []
    labels_all = []

    with torch.no_grad():

        for images, labels in dataloader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            predictions.extend(preds.cpu().tolist())
            labels_all.extend(labels.cpu().tolist())

    epoch_loss = running_loss / len(dataloader.dataset)

    epoch_acc = (
        100.0
        * sum(p == t for p, t in zip(predictions, labels_all))
        / len(labels_all)
    )

    return (
        epoch_loss,
        epoch_acc,
        predictions,
        labels_all,
    )

In [ ]:
# ============================================================
# Stage 2
# Fine-tune the Entire Model
# ============================================================

print("=" * 60)
print("Stage 2")
print(f"Fine-tuning entire {BACKBONE_NAME}")
print("=" * 60)

# ------------------------------------------------------------
# Local checkpoint path
# ------------------------------------------------------------

BEST_MODEL_PATH = f"/content/best_stage2_{BACKBONE_NAME}.pth"

# ------------------------------------------------------------
# Unfreeze entire model
# ------------------------------------------------------------

for param in model.parameters():
    param.requires_grad = True

# ------------------------------------------------------------
# Get parameter groups
# ------------------------------------------------------------

backbone_params, head_params = get_parameter_groups(
    model,
    BACKBONE_NAME
)

# ------------------------------------------------------------
# Discriminative learning rates
# ------------------------------------------------------------

BACKBONE_LR = 1e-5
HEAD_LR = 1e-4

optimizer = torch.optim.Adam(
    [
        {
            "params": backbone_params,
            "lr": BACKBONE_LR,
        },
        {
            "params": head_params,
            "lr": HEAD_LR,
        },
    ]
)

# ------------------------------------------------------------
# Scheduler
# ------------------------------------------------------------

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30
)

criterion = nn.CrossEntropyLoss()

stage2_epochs = 30

best_stage2_acc = stage1_best_acc
best_stage2_epoch = stage1_best_epoch

print("\nStage 2 Configuration")
print("-" * 40)
print(f"Backbone LR : {BACKBONE_LR}")
print(f"Head LR     : {HEAD_LR}")
print(f"Epochs      : {stage2_epochs}")
print("Scheduler   : CosineAnnealingLR")
print("Loss        : CrossEntropyLoss")

print("\nParameter Groups")
print("-" * 40)
print(f"Backbone Parameters : {sum(p.numel() for p in backbone_params):,}")
print(f"Head Parameters     : {sum(p.numel() for p in head_params):,}")

print(f"\n✓ Entire {BACKBONE_NAME} unfrozen.")

Stage 2
Fine-tuning entire ViT_B16

Stage 2 Configuration
----------------------------------------
Backbone LR : 1e-05
Head LR     : 0.0001
Epochs      : 30
Scheduler   : CosineAnnealingLR
Loss        : CrossEntropyLoss

Parameter Groups
----------------------------------------
Backbone Parameters : 85,798,656
Head Parameters     : 3,076

✓ Entire ViT_B16 unfrozen.


In [ ]:
# ============================================================
# Stage 2
# Load Best Stage 1 + Fine-tune Entire Model
# ============================================================

import os
import torch
import torch.nn as nn
import torch.optim as optim

print("=" * 60)
print("Stage 2")
print(f"Fine-tuning entire {BACKBONE_NAME}")
print("=" * 60)

# ------------------------------------------------------------
# Load BEST Stage 1 checkpoint
# ------------------------------------------------------------

stage1_checkpoint = os.path.join(
    EXPERIMENT_DIR,
    "Stage1",
    f"best_stage1_{BACKBONE_NAME}.pth"
)

if not os.path.exists(stage1_checkpoint):
    raise FileNotFoundError(
        f"Stage 1 checkpoint not found:\n{stage1_checkpoint}"
    )

checkpoint = torch.load(
    stage1_checkpoint,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

print("\nLoaded Stage 1 checkpoint")
print("-" * 40)
print(f"Stage           : {checkpoint['stage']}")
print(f"Epoch           : {checkpoint['epoch']}")
print(f"Best Validation : {checkpoint['best_val_accuracy']:.2f}%")

# ------------------------------------------------------------
# Local Stage 2 checkpoint
# ------------------------------------------------------------

BEST_STAGE2_PATH = f"/content/best_stage2_{BACKBONE_NAME}.pth"

# ------------------------------------------------------------
# Unfreeze entire model
# ------------------------------------------------------------

for param in model.parameters():
    param.requires_grad = True

# ------------------------------------------------------------
# Parameter groups
# ------------------------------------------------------------

backbone_params, head_params = get_parameter_groups(
    model,
    BACKBONE_NAME
)

# ------------------------------------------------------------
# Hyperparameters
# ------------------------------------------------------------

BACKBONE_LR = 1e-5
HEAD_LR = 1e-4

optimizer = optim.Adam(
    [
        {
            "params": backbone_params,
            "lr": BACKBONE_LR,
        },
        {
            "params": head_params,
            "lr": HEAD_LR,
        },
    ]
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30
)

criterion = nn.CrossEntropyLoss()

stage2_epochs = 30

best_stage2_acc = checkpoint["best_val_accuracy"]
best_stage2_epoch = checkpoint["epoch"]

print("\nStage 2 Configuration")
print("-" * 40)
print(f"Backbone LR     : {BACKBONE_LR}")
print(f"Head LR         : {HEAD_LR}")
print(f"Epochs          : {stage2_epochs}")
print(f"Scheduler       : CosineAnnealingLR")
print(f"Starting Acc    : {best_stage2_acc:.2f}%")

print("\nParameter Groups")
print("-" * 40)
print(f"Backbone Params : {sum(p.numel() for p in backbone_params):,}")
print(f"Head Params     : {sum(p.numel() for p in head_params):,}")

print("\nStarting Stage 2 Fine-Tuning...\n")

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

for epoch in range(stage2_epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
    )

    val_loss, val_acc, preds, labels = validate_one_epoch(
        model,
        val_loader,
        criterion,
    )

    scheduler.step()

    print(
        f"Epoch [{epoch+1:02d}/{stage2_epochs}] | "
        f"Train Loss {train_loss:.4f} | "
        f"Train Acc {train_acc:.2f}% | "
        f"Val Loss {val_loss:.4f} | "
        f"Val Acc {val_acc:.2f}%"
    )

    # --------------------------------------------------------
    # Save best checkpoint
    # --------------------------------------------------------

    if val_acc > best_stage2_acc:

        best_stage2_acc = val_acc
        best_stage2_epoch = epoch + 1

        torch.save(
            {
                "stage": 2,
                "epoch": best_stage2_epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_accuracy": best_stage2_acc,
                "model_name": BACKBONE_NAME,
                "model_id": MODEL_ID,
                "library": "timm",
                "pretrained": True,
                "pretrained_dataset": "ImageNet-1K",
                "num_classes": 4,
                "config": {
                    "backbone_lr": BACKBONE_LR,
                    "head_lr": HEAD_LR,
                    "batch_size": BATCH_SIZE,
                    "epochs": stage2_epochs,
                },
            },
            BEST_STAGE2_PATH,
        )

        print(
            f"✅ New best model saved "
            f"(Epoch {best_stage2_epoch} | "
            f"Val Acc: {best_stage2_acc:.2f}%)"
        )

print("\n" + "=" * 60)
print("Stage 2 Finished")
print("=" * 60)
print(f"Best Accuracy : {best_stage2_acc:.2f}%")
print(f"Best Epoch    : {best_stage2_epoch}")
print(f"Local Model   : {BEST_STAGE2_PATH}")

Stage 2
Fine-tuning entire ViT_B16

Loaded Stage 1 checkpoint
----------------------------------------
Stage           : 1
Epoch           : 21
Best Validation : 89.17%

Stage 2 Configuration
----------------------------------------
Backbone LR     : 1e-05
Head LR         : 0.0001
Epochs          : 30
Scheduler       : CosineAnnealingLR
Starting Acc    : 89.17%

Parameter Groups
----------------------------------------
Backbone Params : 85,798,656
Head Params     : 3,076

Starting Stage 2 Fine-Tuning...

Epoch [01/30] | Train Loss 0.1350 | Train Acc 95.38% | Val Loss 0.2079 | Val Acc 92.10%
✅ New best model saved (Epoch 1 | Val Acc: 92.10%)
Epoch [02/30] | Train Loss 0.0781 | Train Acc 97.42% | Val Loss 0.1572 | Val Acc 94.26%
✅ New best model saved (Epoch 2 | Val Acc: 94.26%)
Epoch [03/30] | Train Loss 0.0539 | Train Acc 98.27% | Val Loss 0.1207 | Val Acc 94.96%
✅ New best model saved (Epoch 3 | Val Acc: 94.96%)
Epoch [04/30] | Train Loss 0.0362 | Train Acc 98.70% | Val Loss 0.2070 | 

In [ ]:
import os
import shutil
import torch
from datetime import datetime

# ============================================================
# Save Stage 2 Results to Google Drive
# ============================================================

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

LOCAL_MODEL = f"/content/best_stage2_{BACKBONE_NAME}.pth"

SAVE_DIR = os.path.join(
    EXPERIMENT_DIR,
    "Stage2"
)
os.makedirs(SAVE_DIR, exist_ok=True)

checkpoint_filename = f"best_stage2_{BACKBONE_NAME}.pth"
weights_filename = f"stage2_weights_{BACKBONE_NAME}.pth"
summary_filename = f"stage2_summary_{BACKBONE_NAME}.txt"

checkpoint_path = os.path.join(
    SAVE_DIR,
    checkpoint_filename
)

weights_path = os.path.join(
    SAVE_DIR,
    weights_filename
)

summary_path = os.path.join(
    SAVE_DIR,
    summary_filename
)

# ------------------------------------------------------------
# Copy best checkpoint
# ------------------------------------------------------------

if not os.path.exists(LOCAL_MODEL):
    raise FileNotFoundError(
        f"Checkpoint not found:\n{LOCAL_MODEL}"
    )

shutil.copy2(
    LOCAL_MODEL,
    checkpoint_path
)

print("✅ Best Stage 2 checkpoint copied.")

# ------------------------------------------------------------
# Save weights only
# ------------------------------------------------------------

torch.save(
    model.state_dict(),
    weights_path
)

print("✅ Model weights saved.")

# ------------------------------------------------------------
# Generate experiment report
# ------------------------------------------------------------

summary = f"""
=========================================================
HER2-IHC-40x
Stage 2 Training Summary
=========================================================

Date
---------------------------------------------------------
{datetime.now()}

Experiment
---------------------------------------------------------
Backbone              : {BACKBONE_NAME}
Model ID              : {MODEL_ID}
Library               : timm
Pretrained            : True
Pretrained Dataset    : ImageNet-1K
Number of Classes     : 4

Training Strategy
---------------------------------------------------------
Stage 1               : Frozen backbone
Stage 2               : Fine-tune entire network

Dataset
---------------------------------------------------------
Training Images       : {len(train_dataset)}
Validation Images     : {len(val_dataset)}
Classes               : {", ".join(train_dataset.get_class_names())}

Optimization
---------------------------------------------------------
Optimizer             : Adam
Backbone LR           : {BACKBONE_LR}
Head LR               : {HEAD_LR}
Scheduler             : CosineAnnealingLR
Loss                  : CrossEntropyLoss
Batch Size            : {BATCH_SIZE}
Epochs                : {stage2_epochs}

Results
---------------------------------------------------------
Best Validation Acc   : {best_stage2_acc:.4f} %
Best Epoch            : {best_stage2_epoch}

Saved Files
---------------------------------------------------------
Checkpoint            : {checkpoint_filename}
Weights               : {weights_filename}

Location
---------------------------------------------------------
{SAVE_DIR}

=========================================================
"""

with open(summary_path, "w") as f:
    f.write(summary)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("Stage 2 artifacts saved successfully!")
print("=" * 60)

print(f"Checkpoint : {checkpoint_path}")
print(f"Weights    : {weights_path}")
print(f"Summary    : {summary_path}")
print(f"Best Acc   : {best_stage2_acc:.2f}%")
print(f"Best Epoch : {best_stage2_epoch}")

In [ ]:
# ============================================================
# Final Evaluation (Best Stage 2 Model)
# ============================================================

import os
import csv
import torch
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

# ------------------------------------------------------------
# Load Best Stage 2 Checkpoint
# ------------------------------------------------------------

stage2_checkpoint = os.path.join(
    EXPERIMENT_DIR,
    "Stage2",
    f"best_stage2_{BACKBONE_NAME}.pth"
)

if not os.path.exists(stage2_checkpoint):
    raise FileNotFoundError(stage2_checkpoint)

checkpoint = torch.load(
    stage2_checkpoint,
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])

model.to(device)
model.eval()

print("=" * 60)
print(f"Final Evaluation - {BACKBONE_NAME}")
print("=" * 60)
print(f"Checkpoint : {stage2_checkpoint}")
print(f"Best Epoch : {checkpoint['epoch']}")
print(f"Best Val Accuracy : {checkpoint['best_val_accuracy']:.2f}%")
print("=" * 60)

# ------------------------------------------------------------
# Inference
# ------------------------------------------------------------

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)

        preds = outputs.argmax(dim=1)

        all_predictions.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

metrics = {
    "Backbone": BACKBONE_NAME,
    "Model ID": MODEL_ID,
    "Accuracy": accuracy * 100,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1,
    "Best Epoch": checkpoint["epoch"],
}

print("\nOverall Metrics")
print("-" * 40)

for key, value in metrics.items():

    if isinstance(value, float):
        print(f"{key:<12}: {value:.4f}")
    else:
        print(f"{key:<12}: {value}")

# ------------------------------------------------------------
# Classification Report
# ------------------------------------------------------------

class_names = train_dataset.get_class_names()

report = classification_report(
    all_labels,
    all_predictions,
    target_names=class_names,
    digits=4,
    zero_division=0
)

print("\nClassification Report")
print("-" * 40)
print(report)

# ------------------------------------------------------------
# Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("\nConfusion Matrix")
print("-" * 40)
print(cm)

# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

RESULTS_DIR = os.path.join(
    EXPERIMENT_DIR,
    "Results"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

# ---------------- TXT Report ----------------

txt_file = os.path.join(
    RESULTS_DIR,
    f"evaluation_{BACKBONE_NAME}.txt"
)

with open(txt_file, "w") as f:

    f.write("=" * 60 + "\n")
    f.write(f"Model : {BACKBONE_NAME}\n")
    f.write(f"Model ID : {MODEL_ID}\n")
    f.write("=" * 60 + "\n\n")

    for key, value in metrics.items():

        if isinstance(value, float):
            f.write(f"{key:<15}: {value:.4f}\n")
        else:
            f.write(f"{key:<15}: {value}\n")

    f.write("\nClassification Report\n")
    f.write("-" * 40 + "\n")
    f.write(report)

    f.write("\n\nConfusion Matrix\n")
    f.write("-" * 40 + "\n")
    f.write(np.array2string(cm))

# ---------------- CSV Comparison ----------------

csv_file = os.path.join(
    "/content/drive/MyDrive/HER2_Checkpoints",
    "experiment_results.csv"
)

file_exists = os.path.exists(csv_file)

with open(csv_file, "a", newline="") as f:

    writer = csv.writer(f)

    if not file_exists:

        writer.writerow([
            "Backbone",
            "Model ID",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "Best Epoch",
        ])

    writer.writerow([
        BACKBONE_NAME,
        MODEL_ID,
        round(metrics["Accuracy"], 4),
        round(metrics["Precision"], 4),
        round(metrics["Recall"], 4),
        round(metrics["F1 Score"], 4),
        metrics["Best Epoch"],
    ])

print("\n" + "=" * 60)
print("Evaluation completed successfully!")
print("=" * 60)

print(f"TXT Report : {txt_file}")
print(f"CSV Table  : {csv_file}")

In [ ]:
# ============================================================
# Final Evaluation (Best Stage 2 Model)
# ============================================================

import os
import csv
import torch
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

# ------------------------------------------------------------
# Load Best Stage 2 Checkpoint
# ------------------------------------------------------------

stage2_checkpoint = os.path.join(
    EXPERIMENT_DIR,
    "Stage2",
    f"best_stage2_{BACKBONE_NAME}.pth"
)

if not os.path.exists(stage2_checkpoint):
    raise FileNotFoundError(stage2_checkpoint)

checkpoint = torch.load(
    stage2_checkpoint,
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])

model.to(device)
model.eval()

print("=" * 60)
print(f"Final Evaluation - {BACKBONE_NAME}")
print("=" * 60)
print(f"Checkpoint : {stage2_checkpoint}")
print(f"Best Epoch : {checkpoint['epoch']}")
print(f"Best Val Accuracy : {checkpoint['best_val_accuracy']:.2f}%")
print("=" * 60)

# ------------------------------------------------------------
# Inference
# ------------------------------------------------------------

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)

        preds = outputs.argmax(dim=1)

        all_predictions.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

metrics = {
    "Backbone": BACKBONE_NAME,
    "Model ID": MODEL_ID,
    "Accuracy": accuracy * 100,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1,
    "Best Epoch": checkpoint["epoch"],
}

print("\nOverall Metrics")
print("-" * 40)

for key, value in metrics.items():

    if isinstance(value, float):
        print(f"{key:<12}: {value:.4f}")
    else:
        print(f"{key:<12}: {value}")

# ------------------------------------------------------------
# Classification Report
# ------------------------------------------------------------

class_names = train_dataset.get_class_names()

report = classification_report(
    all_labels,
    all_predictions,
    target_names=class_names,
    digits=4,
    zero_division=0
)

print("\nClassification Report")
print("-" * 40)
print(report)

# ------------------------------------------------------------
# Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("\nConfusion Matrix")
print("-" * 40)
print(cm)

# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

RESULTS_DIR = os.path.join(
    EXPERIMENT_DIR,
    "Results"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

# ---------------- TXT Report ----------------

txt_file = os.path.join(
    RESULTS_DIR,
    f"evaluation_{BACKBONE_NAME}.txt"
)

with open(txt_file, "w") as f:

    f.write("=" * 60 + "\n")
    f.write(f"Model : {BACKBONE_NAME}\n")
    f.write(f"Model ID : {MODEL_ID}\n")
    f.write("=" * 60 + "\n\n")

    for key, value in metrics.items():

        if isinstance(value, float):
            f.write(f"{key:<15}: {value:.4f}\n")
        else:
            f.write(f"{key:<15}: {value}\n")

    f.write("\nClassification Report\n")
    f.write("-" * 40 + "\n")
    f.write(report)

    f.write("\n\nConfusion Matrix\n")
    f.write("-" * 40 + "\n")
    f.write(np.array2string(cm))

# ---------------- CSV Comparison ----------------

csv_file = os.path.join(
    "/content/drive/MyDrive/HER2_Checkpoints",
    "experiment_results.csv"
)

file_exists = os.path.exists(csv_file)

with open(csv_file, "a", newline="") as f:

    writer = csv.writer(f)

    if not file_exists:

        writer.writerow([
            "Backbone",
            "Model ID",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "Best Epoch",
        ])

    writer.writerow([
        BACKBONE_NAME,
        MODEL_ID,
        round(metrics["Accuracy"], 4),
        round(metrics["Precision"], 4),
        round(metrics["Recall"], 4),
        round(metrics["F1 Score"], 4),
        metrics["Best Epoch"],
    ])

print("\n" + "=" * 60)
print("Evaluation completed successfully!")
print("=" * 60)

print(f"TXT Report : {txt_file}")
print(f"CSV Table  : {csv_file}")

In [ ]:
# ============================================================
# Load Best Stage 2 Model
# ============================================================

import os
import torch

stage2_model_path = os.path.join(
    EXPERIMENT_DIR,
    "Stage2",
    f"best_stage2_{BACKBONE_NAME}.pth"
)

if not os.path.exists(stage2_model_path):
    raise FileNotFoundError(stage2_model_path)

checkpoint = torch.load(stage2_model_path, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("=" * 60)
print(f"Loaded Best {BACKBONE_NAME} Stage 2 Model")
print("=" * 60)
print(f"Best Validation Accuracy : {checkpoint['best_val_accuracy']:.2f}%")
print(f"Saved Epoch              : {checkpoint['epoch']}")
print("="*60)

In [ ]:
# ============================================================
# Evaluate on Test Set
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import numpy as np

all_preds = []
all_labels = []

model.eval()

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(
    all_labels,
    all_preds,
    average="weighted"
)

recall = recall_score(
    all_labels,
    all_preds,
    average="weighted"
)

f1 = f1_score(
    all_labels,
    all_preds,
    average="weighted"
)

cm = confusion_matrix(all_labels, all_preds)

print("="*60)
print("Final Test Results")
print("="*60)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nClassification Report\n")

print(classification_report(
    all_labels,
    all_preds,
    target_names=train_dataset.classes
))

In [ ]:
# ============================================================
# Confusion Matrix
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------------------------
# Normalize confusion matrix (percentage)
# ------------------------------------------------------------

cm_normalized = cm.astype(np.float64) / cm.sum(axis=1, keepdims=True)

# Annotation:
# Count
# (Percentage)

annotations = np.empty_like(cm, dtype=object)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        annotations[i, j] = (
            f"{cm[i, j]}\n"
            f"({cm_normalized[i, j]*100:.1f}%)"
        )

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=annotations,
    fmt="",
    cmap="Blues",
    linewidths=0.5,
    square=True,
    cbar=True,
    xticklabels=train_dataset.get_class_names(),
    yticklabels=train_dataset.get_class_names(),
)

plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.title(
    f"Confusion Matrix - {BACKBONE_NAME}",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(rotation=0)
plt.yticks(rotation=0)

plt.tight_layout()

# ------------------------------------------------------------
# Save Figure
# ------------------------------------------------------------

RESULTS_DIR = os.path.join(
    EXPERIMENT_DIR,
    "Results"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

figure_path = os.path.join(
    RESULTS_DIR,
    f"confusion_matrix_{BACKBONE_NAME}.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

print(f"✅ Confusion matrix saved to:\n{figure_path}")

plt.show()

In [ ]:
# ============================================================
# Save Complete Experiment Summary
# ============================================================

import os
from datetime import datetime

SUMMARY_PATH = os.path.join(
    EXPERIMENT_DIR,
    f"experiment_summary_{BACKBONE_NAME}.txt"
)

with open(SUMMARY_PATH, "w") as f:

    f.write("=" * 70 + "\n")
    f.write("HER2-IHC-40x Experiment Summary\n")
    f.write("=" * 70 + "\n\n")

    # --------------------------------------------------------
    # General Information
    # --------------------------------------------------------

    f.write("Date\n")
    f.write("-" * 40 + "\n")
    f.write(f"{datetime.now()}\n\n")

    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    f.write("Dataset\n")
    f.write("-" * 40 + "\n")
    f.write("Dataset : HER2-IHC-40x (WSI Split)\n")
    f.write(f"Training Images : {len(train_dataset)}\n")
    f.write(f"Testing Images  : {len(val_dataset)}\n")
    f.write(f"Number of Classes : {train_dataset.get_num_classes()}\n")
    f.write(f"Class Names : {', '.join(train_dataset.get_class_names())}\n\n")

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    f.write("Model\n")
    f.write("-" * 40 + "\n")
    f.write(f"Backbone : {BACKBONE_NAME}\n")
    f.write(f"Model ID : {MODEL_ID}\n")
    f.write("Library : timm\n")
    f.write("Pretrained : ImageNet-1K\n")
    f.write("Input Size : 224 x 224\n\n")

    # --------------------------------------------------------
    # Stage 1
    # --------------------------------------------------------

    f.write("Stage 1\n")
    f.write("-" * 40 + "\n")
    f.write("Training Strategy : Frozen Backbone\n")
    f.write("Trainable Layers : Classification Head Only\n")
    f.write(f"Epochs : {stage1_epochs}\n")
    f.write("Optimizer : Adam\n")
    f.write("Learning Rate : 1e-4\n")
    f.write("Scheduler : CosineAnnealingLR\n")
    f.write(f"Best Epoch : {best_epoch}\n")
    f.write(f"Best Validation Accuracy : {best_acc:.2f}%\n")
    f.write(
        f"Checkpoint : "
        f"best_stage1_{BACKBONE_NAME}.pth\n\n"
    )

    # --------------------------------------------------------
    # Stage 2
    # --------------------------------------------------------

    f.write("Stage 2\n")
    f.write("-" * 40 + "\n")
    f.write("Training Strategy : Full Fine-Tuning\n")
    f.write(f"Epochs : {stage2_epochs}\n")
    f.write("Optimizer : Adam\n")
    f.write(f"Backbone LR : {BACKBONE_LR}\n")
    f.write(f"Head LR : {HEAD_LR}\n")
    f.write("Scheduler : CosineAnnealingLR\n")
    f.write(f"Best Epoch : {best_stage2_epoch}\n")
    f.write(f"Best Validation Accuracy : {best_stage2_acc:.2f}%\n")
    f.write(
        f"Checkpoint : "
        f"best_stage2_{BACKBONE_NAME}.pth\n\n"
    )

    # --------------------------------------------------------
    # Data Augmentation
    # --------------------------------------------------------

    f.write("Data Augmentation\n")
    f.write("-" * 40 + "\n")
    f.write("Resize(224,224)\n")
    f.write("RandomHorizontalFlip\n")
    f.write("RandomVerticalFlip\n")
    f.write("RandomRotation(10°)\n")
    f.write("Normalize(ImageNet)\n\n")

    # --------------------------------------------------------
    # Final Metrics
    # --------------------------------------------------------

    f.write("Final Evaluation\n")
    f.write("-" * 40 + "\n")
    f.write(f"Accuracy  : {accuracy*100:.2f}%\n")
    f.write(f"Precision : {precision:.4f}\n")
    f.write(f"Recall    : {recall:.4f}\n")
    f.write(f"F1 Score  : {f1:.4f}\n\n")

    # --------------------------------------------------------
    # Files
    # --------------------------------------------------------

    f.write("Generated Files\n")
    f.write("-" * 40 + "\n")
    f.write(f"best_stage1_{BACKBONE_NAME}.pth\n")
    f.write(f"best_stage2_{BACKBONE_NAME}.pth\n")
    f.write(f"evaluation_{BACKBONE_NAME}.txt\n")
    f.write(f"confusion_matrix_{BACKBONE_NAME}.png\n")
    f.write(f"stage1_summary_{BACKBONE_NAME}.txt\n")
    f.write(f"stage2_summary_{BACKBONE_NAME}.txt\n")
    f.write(f"experiment_summary_{BACKBONE_NAME}.txt\n")

print("=" * 60)
print("✅ Experiment summary saved successfully!")
print("=" * 60)
print(SUMMARY_PATH)

In [ ]:
# ============================================================
# Save Confusion Matrix
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix

# ------------------------------------------------------------
# Results Directory
# ------------------------------------------------------------

RESULTS_DIR = os.path.join(
    EXPERIMENT_DIR,
    "Results"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# Compute confusion matrix
# (Uses predictions from Final Evaluation)
# ------------------------------------------------------------

cm = confusion_matrix(
    all_labels,
    all_predictions
)

# Normalize per class

cm_normalized = cm.astype(np.float64)
cm_normalized /= cm.sum(axis=1, keepdims=True)

# Annotation:
# Count
# (Percentage)

annotations = np.empty_like(cm, dtype=object)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):

        annotations[i, j] = (
            f"{cm[i, j]}\n"
            f"({cm_normalized[i, j]*100:.1f}%)"
        )

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=annotations,
    fmt="",
    cmap="Blues",
    linewidths=0.5,
    square=True,
    cbar=True,
    xticklabels=train_dataset.get_class_names(),
    yticklabels=train_dataset.get_class_names(),
)

plt.title(
    f"Confusion Matrix - {BACKBONE_NAME}",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)

plt.xticks(rotation=0)
plt.yticks(rotation=0)

plt.tight_layout()

# ------------------------------------------------------------
# Save Figure
# ------------------------------------------------------------

figure_path = os.path.join(
    RESULTS_DIR,
    f"confusion_matrix_{BACKBONE_NAME}.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("=" * 60)
print("✅ Confusion Matrix Saved")
print("=" * 60)
print(f"Model : {BACKBONE_NAME}")
print(f"Saved : {figure_path}")

In [ ]:
# ============================================================
# Save Classification Report
# ============================================================

import os
import pandas as pd

from sklearn.metrics import classification_report

# ------------------------------------------------------------
# Results Directory
# ------------------------------------------------------------

RESULTS_DIR = os.path.join(
    EXPERIMENT_DIR,
    "Results"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# Classification Report
# (Uses predictions from Final Evaluation)
# ------------------------------------------------------------

class_names = train_dataset.get_class_names()

report_text = classification_report(
    all_labels,
    all_predictions,
    target_names=class_names,
    digits=4,
    zero_division=0
)

report_dict = classification_report(
    all_labels,
    all_predictions,
    target_names=class_names,
    digits=4,
    zero_division=0,
    output_dict=True
)

# ------------------------------------------------------------
# Display Report
# ------------------------------------------------------------

print("=" * 60)
print(f"Classification Report - {BACKBONE_NAME}")
print("=" * 60)

print(report_text)

# ------------------------------------------------------------
# Save TXT Report
# ------------------------------------------------------------

txt_path = os.path.join(
    RESULTS_DIR,
    f"classification_report_{BACKBONE_NAME}.txt"
)

with open(txt_path, "w") as f:

    f.write("=" * 60 + "\n")
    f.write(f"Classification Report\n")
    f.write("=" * 60 + "\n\n")

    f.write(f"Backbone : {BACKBONE_NAME}\n")
    f.write(f"Model ID : {MODEL_ID}\n")
    f.write(f"Best Validation Accuracy : {checkpoint['best_val_accuracy']:.2f}%\n\n")

    f.write(report_text)

# ------------------------------------------------------------
# Save CSV Report
# ------------------------------------------------------------

csv_path = os.path.join(
    RESULTS_DIR,
    f"classification_report_{BACKBONE_NAME}.csv"
)

report_df = pd.DataFrame(report_dict).transpose()

report_df.to_csv(
    csv_path,
    index=True
)

print("\n" + "=" * 60)
print("✅ Classification Report Saved")
print("=" * 60)
print(f"TXT : {txt_path}")
print(f"CSV : {csv_path}")